kernal: scanpy

# Set up

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as mpdf
from matplotlib.pyplot import rc_context

import scanpy as sc
import muon as mu

import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.filterwarnings(action='once')
warnings.simplefilter(action='once')
warnings.simplefilter(action="ignore", category=NumbaDeprecationWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)

In [ ]:
sc.settings.verbosity = 0  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(8, 7), facecolor="white")
sc.logging.print_versions()

In [ ]:
blood_markers_dict = {
    "HSC": ["CD34", "SPINK2", "MLLT3", "HLF", "MECOM", "RUNX1", "HOXA9", 
            "CDK6", "SELL", "CD52", "PROM1", "MEIS1", "MYB", "ITGA6"],
    "GP": ["MPO", "AZU1", "SPI1", "LYZ"],
    "MEMP": ["GATA1", "GATA2", "TESPA1", "KLF1", "CTNNB1"],
    "Ery": ["FAM178B", "TFRC", "AHSP", "ALAS2", "HBA1", "HBB", "GYPA", "BPGM"],
    "MK": ["ITGA2B", "GP9", "PLEK", 'MPL', 'PECAM1', 'CXCR4', "PPBP", "PF4"],
    "Mast": ["HDC", "CPA3", "LMO4", "CD63", 'ENPP3', "TPSAB1", "TPSB2"],
    "Mono": ["CD14", "FCGR3A", "FCN1", "VCAN", "S100A9", "CD68", "MNDA"],
    "Kupffer": ["CD163", "MS4A7", "C1QA", "MRC1", "CTSB", "MARCO", "CD5L", "VCAM1"],
    "cDC1": ["CLEC9A", "THBD", "XCR1", "BATF3"],
    "cDC2": ["CD1C", "CLEC4A", "CLEC10A"],
    "cDC3": ["FLT3", "VCAN", "CD14", "S100A8"],
    "pDC": ["IRF8", "CLEC4C", "IL3RA", "MPEG1"],
    "ASDC": ["AXL", "SIGLEC6"],
    "LP": ["IL7R", "JCHAIN", "LTB", "CD7"],
    "B": ["EBF1", "PAX5", "CD79A", "CD79B", "MME", "IGLL1", "IGHM", "IGHD", "CD19", "MS4A1", "IRF4", "DNTT", "RAG1", "RAG2", "CD24", "CD38"],
    "B1": ["CD5", "CD27", "SPN", "CCR10"],
    "NK": ["IL2RB", "KLRD1", "KLRF1", "NCR1", "NCAM1", "FCGR3A"],
    "ILC": ["TCF7", "RORC", "AHR", "ID2", "NCR2"],
    "T": ["CD3D", "CD3E", "CD3G", "TRAC", "FOXP3", "TIGIT", "CD4", "CD8A", "CD8B"],
    "Mix": ["KIT", "GATA3", "IL1A", "IL1B", "PTPRC"],
    "Hepa": ["ALB", "AFP"],
    "Endo": ["CDH5", "KDR"],
    "LSEC": ["STAB1", "STAB2", "LYVE1"],
    "Stellate": ["DCN", "COL1A1", "COL3A1", "RBP1"],
    "Epi": ["KRT19"],
    "Cycling": ["MKI67", "TOP2A"]
}

blood_markers_lst = [
  "CD34", "SPINK2", "MLLT3", "HLF", "MECOM", "RUNX1", "HOXA9", 
  "CDK6", "SELL", "CD52", "PROM1", "MEIS1", "MYB", "ITGA6", # HSC/MPP
  "MPO", "AZU1", "SPI1", "LYZ", # Granulocyte
  "GATA1", "GATA2", "TESPA1", "KLF1", "CTNNB1", # MEMPs (megakaryocyte-erythroid-mast cell progenitor)
  "FAM178B", "TFRC", "AHSP", "ALAS2", "HBA1", "HBB", "GYPA", "BPGM", # Erythroid
  "ITGA2B", "GP9", "PLEK", 'MPL', 'PECAM1', 'CXCR4', "PPBP", "PF4", # Megakaryocytes
  "HDC", "CPA3", "LMO4", "CD63", 'ENPP3', "TPSAB1", "TPSB2", # Mast cells
  "CD14", "FCGR3A", "FCN1", "S100A9", "CD68", "MNDA", # Monocytes
  "CD163", "MS4A7", "C1QA", "MRC1", "CTSB", "MARCO", "CD5L", "VCAM1", # Kupffer cells
  "CLEC9A", "THBD", "XCR1", "BATF3", # cDC1
  "CD1C", "CLEC4A", "CLEC10A", # cDC2
  "FLT3", "VCAN", # cDC3
  "IRF8", "CLEC4C", "IL3RA", "MPEG1", # pDCs
  "AXL", "SIGLEC6", # ASDC
  "IL7R", "JCHAIN", "LTB", "CD7", # LP
  "EBF1", "PAX5", "CD79A", "CD79B", "MME", "IGLL1", "IGHM", "IGHD",
  "CD19", "MS4A1", "IRF4", "DNTT", "RAG1", "RAG2", "CD24", "CD38", # B cells
  "CD5", "CD27", "SPN", "CCR10", # B1
  "IL2RG", "NKG7", "PRF1", "GZMA", "KLRB1", "TRBC1", "IL2RB", # NK
  "TCF7", "RORC", "AHR", "ID2", "NCR2", # ILC3
  "CD3D", "CD3E", "CD3G", "FOXP3", "CD4", "CD8A", "CD8B", # T
  "KIT", "GATA3", "IL1A", "IL1B",
  "PTPRC", # CD45
  "ALB", "AFP", # Hepatocytes
  "CDH5", "KDR", # endothelial cells
  "STAB1", "STAB2", "LYVE1", "DCN", # LSECs
  "COL1A1", "COL3A1", "RBP1", # stellate cells
  "KRT19",
  'MKI67', "TOP2A" # cycling
]

HSC_sanity_check = {
    "HSC": ["CD34", "SPINK2", "MLLT3", "HLF", "MECOM", "RUNX1", "HOXA9", 
            "CDK6", "SELL", "CD52", "PROM1", "MEIS1", "MYB", "ITGA6"],
    "GP": ["MPO", "AZU1", "SPI1", "LYZ"],
    "MEMP": ["GATA1", "GATA2", "TESPA1", "KLF1", "CTNNB1"],
    "LP": ["IL7R", "JCHAIN", "LTB", "CD7", "IL2RG"],
    "B-lin": ["EBF1", "PAX5", "CD19"],
    "ILC": ["TCF7", "RORC"],
    "NK/T-lin": ["IL2RB", "KLRD1", "CD3D", "BCL11B"],
    "Cycling": ["MKI67", "TOP2A"]
}

# Load data

In [ ]:
work_dir = '/work/DevM_analysis/02.abundance/Milo_FL_HSC'
dataset = "FL_wnn"
new_file, old_file = "v00", "v00"
new_anno, old_anno = "anno_wnn_hsc_sub", "anno_wnn_hsc_sub"

Load data

In [ ]:
mdata = mu.read(
    f"data/{dataset}_clustered.{old_file}.h5mu"
)
mdata

# Dotplot

leiden_wnn_0.3

In [ ]:
mdata.obs[new_anno].value_counts()

In [ ]:
sc.pl.dotplot(mdata['rna'], var_names=blood_markers_dict, groupby=[new_anno], standard_scale="var",
             show=False, figsize=(45, len(mdata.obs[new_anno].cat.categories) * 0.35))

In [ ]:
sc.pl.dotplot(mdata['rna'], var_names=HSC_sanity_check, groupby=[new_anno], standard_scale="var",
             show=False, figsize=(12, len(mdata.obs[new_anno].cat.categories) * 0.4))

# UMAP

Random cells

In [ ]:
np.random.seed(0)
random_indices = np.random.permutation(list(range(mdata.shape[0])))

## Cluster

Leiden res. 0.3

In [ ]:
with rc_context({"figure.figsize": (11, 10)}):
    mu.pl.embedding(mdata[random_indices, :], basis='umap', color=[new_anno],
                size=10, legend_loc="on data", show=False)
    plt.savefig(f"{work_dir}/plots/{dataset}_umap_cluster.{new_file}.pdf", bbox_inches="tight")

Library/Donor

In [ ]:
with plt.rc_context({"figure.figsize": (12, 9)}):  # Use this to set figure params like size and dpi
    mu.pl.embedding(mdata[random_indices, :], basis='umap', color=["libraryID"], size=10, show=False)
    plt.savefig(f"{work_dir}/plots/{dataset}_umap_library.{new_file}.pdf", bbox_inches="tight")

In [ ]:
with plt.rc_context({"figure.figsize": (12, 9)}):  # Use this to set figure params like size and dpi
    mu.pl.embedding(mdata[random_indices, :], basis='umap', color=["donorID"], size=10, show=False)
    plt.savefig(f"{work_dir}/plots/{dataset}_umap_donor.{new_file}.pdf", bbox_inches="tight")

PCW

In [ ]:
with plt.rc_context({"figure.figsize": (12, 9)}):  # Use this to set figure params like size and dpi
    mu.pl.embedding(mdata[random_indices, :], basis='umap', color=["PCW"], size=10, show=False)
    plt.savefig(f"{work_dir}/plots/{dataset}_umap_PCW.{new_file}.pdf", bbox_inches="tight")

Cell cycle

In [ ]:
with plt.rc_context({"figure.figsize": (12, 10)}):  # Use this to set figure params like size and dpi
    mu.pl.embedding(mdata[random_indices, :], basis='umap', color=["rna:Phase", "rna:G2M.Score"], size=10, show=False)
    plt.savefig(f"{work_dir}/plots/{dataset}_umap_cellCycle.{new_file}.pdf", bbox_inches="tight")

# Composition

In [ ]:
d4p = mdata.obs.copy()

In [ ]:
def compo_plot(data=None, groupby=None, condition=None):
    #groupby_key = "leiden_wnn_0.9"
    #condition_key = rna_anno
    df = pd.crosstab(data[groupby], data[condition])
    df = df.div(df.sum(axis=1), axis=0) * 100.0
    ax = df.plot(
            kind = "bar",
            stacked = True,
            legend = False
        )
    ax.set_xlabel(groupby)
    ax.set_ylabel("Percentage")
    ax.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), ncol=3)
    if len(max(df.index.astype(str), key=len)) >= 5:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

Library

In [ ]:
with rc_context({"figure.figsize": (len(mdata.obs[new_anno].cat.categories) * 1, 10)}):
    groupby_key = new_anno
    condition_key = "libraryID"
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_clusterByLibrary.{new_file}.pdf", bbox_inches="tight")

In [ ]:
with rc_context({"figure.figsize": (25, 10)}):
    groupby_key = "libraryID"
    condition_key = new_anno
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_LibraryByCluster.{new_file}.pdf", bbox_inches="tight")

Sample

In [ ]:
with rc_context({"figure.figsize": (len(mdata.obs[new_anno].cat.categories) * 1, 10)}):
    groupby_key = new_anno
    condition_key = "sampleID"
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_clusterBySample.{new_file}.pdf", bbox_inches="tight")

In [ ]:
with rc_context({"figure.figsize": (25, 10)}):
    groupby_key = "sampleID"
    condition_key = new_anno
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_SampleByCluster.{new_file}.pdf", bbox_inches="tight")

Donor

In [ ]:
with rc_context({"figure.figsize": (len(mdata.obs[new_anno].cat.categories) * 1, 10)}):
    groupby_key = new_anno
    condition_key = "donorID"
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_clusterByDonor.{new_file}.pdf", bbox_inches="tight")

In [ ]:
with rc_context({"figure.figsize": (25, 10)}):
    groupby_key = "donorID"
    condition_key = new_anno
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_DonorByCluster.{new_file}.pdf", bbox_inches="tight")

PCW

In [ ]:
with rc_context({"figure.figsize": (len(mdata.obs[new_anno].cat.categories) * 1, 10)}):
    groupby_key = new_anno
    condition_key = "PCW"
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_clusterByPCW.{new_file}.pdf", bbox_inches="tight")

In [ ]:
with rc_context({"figure.figsize": (15, 10)}):
    groupby_key = "PCW"
    condition_key = new_anno
    compo_plot(data=d4p, groupby=groupby_key, condition=condition_key)
    plt.savefig(f"{work_dir}/plots/{dataset}_compositionPlot_PCWByCluster.{new_file}.pdf", bbox_inches="tight")

# QC

## Cluster size, gene/umi/peak count

In [ ]:
df = d4p.groupby(new_anno).agg({"rna:nFeature_RNA": 'median', 'rna:nCount_RNA': 'median', 'atac:nCount_peaks': 'median', 'atac:nFeature_peaks': 'median'})
df['count'] = d4p[new_anno].value_counts()
#df.to_csv(f"{work_dir}/data/{dataset}_clusterSizes_medianCounts.{new_file}.csv")

## Doublet

In [ ]:
my_order = d4p.groupby(by=[new_anno])["rna:scDblFinder.score"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = 'rna:scDblFinder.score', x = new_anno, order=my_order)
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

## RNA

nCount_RNA

In [ ]:
my_order = d4p.groupby(by=[new_anno])["rna:nCount_RNA"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = "rna:nCount_RNA", x = new_anno, order=my_order,)
plt.yscale('log')
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.show()

nFeature_RNA

In [ ]:
my_order = d4p.groupby(by=[new_anno])["rna:nFeature_RNA"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = "rna:nFeature_RNA", x = new_anno, order=my_order, )
plt.yscale('log')
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')

# Add horizontal lines
plt.axhline(y=300, color='red', linestyle='--', linewidth=2)  # Add horizontal line at y=1000
plt.axhline(y=500, color='red', linestyle='-.', linewidth=2)  # Add horizontal line at y=10000

plt.show()

percent.mt

In [ ]:
my_order = d4p.groupby(by=[new_anno])["rna:percent.mt"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = "rna:percent.mt", x = new_anno, order=my_order, )
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

percent.rb

In [ ]:
my_order = d4p.groupby(by=[new_anno])["rna:percent.rb"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = "rna:percent.rb", x = new_anno, order=my_order, )
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

## ATAC

nCount_peaks

In [ ]:
my_order = d4p.groupby(by=[new_anno])["atac:nCount_peaks"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = 'atac:nCount_peaks', x = new_anno, order=my_order, )
plt.yscale('log')
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

nFeature_peaks

In [ ]:
my_order = d4p.groupby(by=[new_anno])["atac:nFeature_peaks"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = 'atac:nFeature_peaks', x = new_anno, order=my_order, )
plt.yscale('log')

p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

nucleosome_signal

In [ ]:
my_order = d4p.groupby(by=[new_anno])["atac:nucleosome_signal"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = 'atac:nucleosome_signal', x = new_anno, order=my_order, )
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

TSS.enrichment

In [ ]:
my_order = d4p.groupby(by=[new_anno])["atac:TSS.enrichment"].median().sort_values().index
plt.figure(figsize=(len(mdata.obs[new_anno].cat.categories) * 0.4, 5))
p = sns.boxplot(data = d4p, y = 'atac:TSS.enrichment', x = new_anno, order=my_order, )
p.set_xticklabels(p.get_xticklabels(), rotation=45, horizontalalignment='center')
plt.show()

# Save

In [ ]:
mdata